# Create BD² Awards

**BD² (Breakthrough Discoveries for Thriving with Bipolar Disorder)** (funder_id
`8901078393`, IAMHRF priority-40, priority `309`). bipolardiscoveries.org program pages
(Discovery Research / Collaboration / Brain Omics / Genetics) via Playwright accordion
expansion (WPE-walled). 18 grants, 100% PI/institution/title, year 89%; no per-grant
amounts at source (§6.7). NOTE: this funder id is absent from openalex.common.funder —
funder struct hardcoded from the live API (registry gap flagged to Kyle).
Parquet: `s3://openalex-ingest/awards/bd2/bd2_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bd2_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bd2/bd2_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be ~18)
SELECT COUNT(*) as total_projects FROM openalex.awards.bd2_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.bd2_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bd2_raw LIMIT 5;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bd2_awards
USING delta
AS
WITH
funder_row AS (
    -- F8901078393 is MISSING from openalex.common.funder (registry gap, flagged);
    -- struct values taken from api.openalex.org/funders/F8901078393 on 2026-06-12
    SELECT CAST(8901078393 AS BIGINT) as funder_id,
           'BD2 Breakthrough Discoveries for thriving with Bipolar Disorder' as display_name,
           'https://ror.org/00z5dw933' as ror_id,
           '10.13039/100028749' as doi
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'bd2' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.bd2_raw g
    CROSS JOIN funder_row f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bd2' AND priority = 309;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    309 as priority
FROM openalex.awards.bd2_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.bd2_awards;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.bd2_awards GROUP BY 1;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(display_name) as has_title, COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi, COUNT(start_year) as has_year
FROM openalex.awards.bd2_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bd2' AND priority = 309;